# Adult Income Prediction Dataset - Proyecto Final

Luciana Hoyos Pérez, Juan José Gómez Vélez y Santiago Manco Maya

## 1. Análisis Preliminar del Problema

Este dataset corresponde a un problema de clasificación, ya que la variable objetivo (target) income indica si un adulto gana más o menos de 50k dólares al año. Es decir, el modelo debe aprender a clasificar a los individuos en dos categorías: altos salarios y bajos salarios. Por lo tanto, no se busca predecir un valor numérico continuo (como en regresión), sino asignar cada observación a una clase definida.



In [45]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
import numpy as np

from sklearn.datasets import fetch_openml
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.metrics import f1_score, ConfusionMatrixDisplay, confusion_matrix
from sklearn.neighbors import KNeighborsClassifier

#import tensorflow as tf
#from tensorflow.keras.models import Sequential
#from tensorflow.keras.layers import Dense, Dropout
#rom tensorflow.keras import regularizers

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import (
    MinMaxScaler,
    OneHotEncoder,
    OrdinalEncoder,
    StandardScaler,
    PolynomialFeatures,
    FunctionTransformer,
)

datos = pd.read_csv("adult.csv")

datos.info()

# Diccionario para clasificar variables
tipos_variables = {
    "numéricas": [],
    "categóricas": [],
    "binarias": [],
    "ordinales": []
}

for col in datos.columns:

    if pd.api.types.is_numeric_dtype(datos[col]):

        if datos[col].nunique() == 2:
            tipos_variables["binarias"].append(col)
        else:
            tipos_variables["numéricas"].append(col)

    else:

        if datos[col].nunique() == 2:
            tipos_variables["binarias"].append(col)
        else:
            tipos_variables["categóricas"].append(col)

tipos_variables["ordinales"] = ["EducationalLevel"]

# Mostrar resultado
for tipo, columnas in tipos_variables.items():
    print(f"\n{tipo.upper()}:")
    for c in columnas:
        print(f" - {c}")

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 32561 entries, 0 to 32560
Data columns (total 15 columns):
 #   Column          Non-Null Count  Dtype 
---  ------          --------------  ----- 
 0   age             32561 non-null  int64 
 1   workclass       32561 non-null  object
 2   fnlwgt          32561 non-null  int64 
 3   education       32561 non-null  object
 4   education.num   32561 non-null  int64 
 5   marital.status  32561 non-null  object
 6   occupation      32561 non-null  object
 7   relationship    32561 non-null  object
 8   race            32561 non-null  object
 9   sex             32561 non-null  object
 10  capital.gain    32561 non-null  int64 
 11  capital.loss    32561 non-null  int64 
 12  hours.per.week  32561 non-null  int64 
 13  native.country  32561 non-null  object
 14  income          32561 non-null  object
dtypes: int64(6), object(9)
memory usage: 3.7+ MB

NUMÉRICAS:
 - age
 - fnlwgt
 - education.num
 - capital.gain
 - capital.loss
 - hours.per.we

Ese dataset proviene originalmente del UCI Machine Learning Repository, bajo el nombre Adult o Census Income dataset.

Más específicamente:

- Los datos provienen del Censo de los EE. UU. de 1994.
- Fueron extraídos por Barry Becker (y otros colaboradores) a partir de registros del censo, aplicando ciertos filtros (por ejemplo: edad >16, horas de trabajo >0, etc.).
- El objetivo original fue generar un conjunto de registros “limpios” para predicción: determinar si el ingreso de una persona excede los $50,000 anuales.

## 2. Análisis exploratorio de datos (EDA) y Limpieza

In [46]:
# Distribución de cada variable

for col in datos.columns:
    print(f"\nDistribución de {col}")
    print(datos[col].value_counts(dropna=False))


Distribución de age
age
36    898
31    888
34    886
23    877
35    876
     ... 
83      6
88      3
85      3
86      1
87      1
Name: count, Length: 73, dtype: int64

Distribución de workclass
workclass
Private             22696
Self-emp-not-inc     2541
Local-gov            2093
?                    1836
State-gov            1298
Self-emp-inc         1116
Federal-gov           960
Without-pay            14
Never-worked            7
Name: count, dtype: int64

Distribución de fnlwgt
fnlwgt
123011    13
203488    13
164190    13
126675    12
121124    12
          ..
222966     1
301229     1
155382     1
268083     1
113987     1
Name: count, Length: 21648, dtype: int64

Distribución de education
education
HS-grad         10501
Some-college     7291
Bachelors        5355
Masters          1723
Assoc-voc        1382
11th             1175
Assoc-acdm       1067
10th              933
7th-8th           646
Prof-school       576
9th               514
12th              433
Doctorate     

In [47]:
# Reemplazar " ?" y espacios vacíos por NaN y eliminar filas con valores faltantes
datos = datos.replace(' ?', pd.NA)
datos = datos.replace(r'^\s*$', pd.NA, regex=True)
datos = datos.dropna()

cat_cols = datos.select_dtypes(include=['object']).columns
num_cols = datos.select_dtypes(exclude=['object']).columns

# Mostrar resultado final
print("\nLimpieza completa.")
print(f"Filas finales: {len(datos)}")
print(f"Columnas finales: {len(datos.columns)}")
print("\nPrimeras filas del dataset transformado:")
print(datos.head())


Limpieza completa.
Filas finales: 32561
Columnas finales: 15

Primeras filas del dataset transformado:
   age workclass  fnlwgt     education  education.num marital.status  \
0   90         ?   77053       HS-grad              9        Widowed   
1   82   Private  132870       HS-grad              9        Widowed   
2   66         ?  186061  Some-college             10        Widowed   
3   54   Private  140359       7th-8th              4       Divorced   
4   41   Private  264663  Some-college             10      Separated   

          occupation   relationship   race     sex  capital.gain  \
0                  ?  Not-in-family  White  Female             0   
1    Exec-managerial  Not-in-family  White  Female             0   
2                  ?      Unmarried  Black  Female             0   
3  Machine-op-inspct      Unmarried  White  Female             0   
4     Prof-specialty      Own-child  White  Female             0   

   capital.loss  hours.per.week native.country income 

In [48]:
# Codificar categóricas
datos_cod = pd.get_dummies(datos, drop_first=True)

# Buscar la variable objetivo codificada (>50K o <=50K)
target_col = [c for c in datos_cod.columns if 'income' in c.lower()][0]

# Calcular correlación con la variable objetivo
correlaciones_objetivo = datos_cod.corr()[target_col].sort_values(ascending=False)

print("\n🎯 Correlación de todas las variables con la variable objetivo:")
print(correlaciones_objetivo)


🎯 Correlación de todas las variables con la variable objetivo:
income_>50K                          1.000000
marital.status_Married-civ-spouse    0.444696
education.num                        0.335154
age                                  0.234037
hours.per.week                       0.229689
                                       ...   
relationship_Unmarried              -0.142857
occupation_Other-service            -0.156348
relationship_Not-in-family          -0.188497
relationship_Own-child              -0.228532
marital.status_Never-married        -0.318440
Name: income_>50K, Length: 101, dtype: float64
